# Training metrics

Plots the health metrics that `ActorCriticAgent.train()` appends to
`../training_metrics.txt` (one line per PPO update, trailing comma stripped):

| column | meaning |
|---|---|
| 1 | update counter (1-based, per process) |
| 2 | mean player/policy loss |
| 3 | mean critic loss (Huber) |
| 4 | mean approximate KL |
| 5 | clip fraction |
| 6 | mean bid reward per update |
| 7 | mean play reward per update |

`../game_meta.txt` has one line per `game_data.txt` line (timestamp, username,
player count, phase1 flag, train flag, agent update counter) and is written
since the metadata change, so the second half of this notebook can split game
progress by phase and separate sampled training games from greedy eval games.

Run at least one game/update with the current code before using this notebook;
`training_metrics.txt` and the `game_meta.txt` columns only exist from then on.

In [ ]:
using Plots, DelimitedFiles, Statistics, RollingFunctions

In [ ]:
metrics = Float64.(readdlm("../training_metrics.txt", ',')[:, 1:end-1])
updates       = @view metrics[:, 1]
player_loss   = @view metrics[:, 2]
critic_loss   = @view metrics[:, 3]
approx_kl     = @view metrics[:, 4]
clip_fraction = @view metrics[:, 5]
bid_reward    = @view metrics[:, 6]
play_reward   = @view metrics[:, 7]
N = min(50, max(1, length(updates) ÷ 10))
# runmean errors when the window exceeds the series length (e.g. a
# short eval subset below), so clamp it and handle empty series.
roll(x) = isempty(x) ? Float64[] : runmean(x, min(N, length(x)))
size(metrics)

In [ ]:
plot(updates, [player_loss runmean(player_loss, N)],
     alpha=[0.3 1], label=["Raw" "runmean($N)"],
     ylabel="Player (policy) loss", xlabel="Update")

In [ ]:
plot(updates, [critic_loss runmean(critic_loss, N)],
     alpha=[0.3 1], label=["Raw" "runmean($N)"],
     ylabel="Critic loss (Huber)", xlabel="Update")

In [ ]:
plot(updates, [approx_kl runmean(approx_kl, N)],
     alpha=[0.3 1], label=["Raw" "runmean($N)"],
     ylabel="Approx KL", xlabel="Update")
hline!([0.02, 0.03], c=:black, ls=:dash, label="")
# dashes: target_kl (0.02) and the early-stop threshold (1.5 * target_kl)

In [ ]:
plot(updates, [clip_fraction runmean(clip_fraction, N)],
     alpha=[0.3 1], label=["Raw" "runmean($N)"],
     ylabel="Clip fraction", xlabel="Update")

In [ ]:
plot(updates, [bid_reward runmean(bid_reward, N)],
     alpha=[0.3 1], label=["Raw" "runmean($N)"],
     ylabel="Bid reward / update (score delta / 50)", xlabel="Update")

In [ ]:
plot(updates, [play_reward runmean(play_reward, N)],
     alpha=[0.3 1], label=["Raw" "runmean($N)"],
     ylabel="Play reward / update", xlabel="Update")

## Game progress split by phase and train flag

`correct` is the per-game correct-bid rate computed directly from
`game_data.txt` (15 bids followed by 15 tricks per player-game). `game_meta.txt`
is aligned on the most recent rows because it only covers games logged after the
metadata change.

In [ ]:
games = Float64.(readdlm("../game_data.txt", ',')[:, 1:end-1])
meta  = readdlm("../game_meta.txt", ',')
n = min(size(games, 1), size(meta, 1))
games = games[end-n+1:end, :]
meta  = meta[end-n+1:end, :]
phase1  = Int.(meta[:, 4])   # 1 = bid head trained by cross entropy
train   = Int.(meta[:, 5])   # 0 = greedy eval game
correct = Float64.(vec(mean(games[:, 1:15] .== games[:, 16:30], dims=2)))
n

In [ ]:
p1 = roll(correct[phase1 .== 1])
p2 = roll(correct[phase1 .== 0])
plt = plot(ylim=(0, 1), ylabel="Correct-bid rate", xlabel="Game")
isempty(p1) || plot!(plt, p1, label="phase 1 (bid cross entropy)")
isempty(p2) || plot!(plt, p2, label="phase 2 (score PPO)")
plt

In [ ]:
pt = roll(correct[train .== 1])
pe = roll(correct[train .== 0])
plt = plot(ylim=(0, 1), ylabel="Correct-bid rate", xlabel="Game")
isempty(pt) || plot!(plt, pt, label="train (sampled)")
isempty(pe) || plot!(plt, pe, label="eval (greedy)")
plt

In [ ]:
#savefig("metrics.png")